# Module 4 — NumPy and Relationship Analysis

**Modular Daraz Analysis — Module 4 of 6**

**Purpose.** Use NumPy and pandas correlations to quantify the sold-count distribution and test which numerical variables move together.

**Course concepts visible in this part:** NumPy arrays, `np.percentile()`, quartiles, IQR, boolean array comparisons, `np.sum()`, `np.corrcoef()`, pandas correlation and interpretation.

**Project rule:** This notebook is one stage of the same Daraz analysis. It does not use separate practice examples; every concept is applied to the real project data.

## Load the analysis-ready output from Module 3
This part focuses only on numerical/statistical analysis. It does not repeat the cleaning or EDA stages.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("daraz_analysis_ready.csv")
print("Analysis-ready shape:", df.shape)

Analysis-ready shape: (1051, 12)


### NumPy numerical analysis

Instead of importing NumPy only for demonstration, we use it on the cleaned **Sold Count** data. NumPy arrays are useful for numerical calculations such as percentiles, spread and boolean comparisons.

Here we calculate the **25th percentile (Q1), median (Q2), 75th percentile (Q3), interquartile range (IQR)** and a standard 1.5×IQR upper threshold. We use this to describe the skew and spread of reported sales; we **do not automatically delete** high-sales products, because they may be legitimate popular listings.

In [2]:
# NumPy: convert cleaned sold-count values into a numerical array
sold_array = df["Sold Count"].dropna().to_numpy(dtype=float)

# quartiles and interquartile range
q1, q2, q3 = np.percentile(sold_array, [25, 50, 75])
iqr = q3 - q1
upper_iqr_threshold = q3 + 1.5 * iqr

# boolean comparison on the NumPy array
high_sales_count = np.sum(sold_array > upper_iqr_threshold)

print("available sold-count records:", sold_array.size)
print("Q1 (25th percentile):", q1)
print("median (50th percentile):", q2)
print("Q3 (75th percentile):", q3)
print("IQR:", iqr)
print("1.5 x IQR upper threshold:", upper_iqr_threshold)
print("records above that threshold:", int(high_sales_count))

available sold-count records: 1020
Q1 (25th percentile): 186.0
median (50th percentile): 443.5
Q3 (75th percentile): 1000.0
IQR: 814.0
1.5 x IQR upper threshold: 2221.0
records above that threshold: 97


**What NumPy adds to the analysis.** The middle 50% of available reported sold counts lies between **186 and 1000**, with a median of **443.5**. The IQR is **814**. There are **97** records above the standard 1.5×IQR upper threshold of **2221**.

This supports what we see later in the visualizations: sold count is strongly right-skewed. We keep these high values because the dataset describes top-selling marketplace products and there is no evidence that those observations are errors.

### Relationship checks

Category summaries tell us **what differs**, but correlations help answer **which numerical variables move together**. We first use pandas to compare each numerical measure with `Sold Count`, then use NumPy to independently verify the strongest correlation.

A correlation near 0 means little linear relationship; values closer to 1 or -1 indicate a stronger positive or negative linear relationship. Correlation does not prove causation. In particular, `Rating Count` naturally grows as more customers buy and review a product, so it should be interpreted as an engagement/popularity measure rather than a cause of sales.

In [3]:
# correlations with reported sold count
relationship_corr = df[[
    "Current Price", "Rating in Stars", "Rating Count", "Discount %", "Sold Count"
]].corr()["Sold Count"].drop("Sold Count").sort_values(ascending=False)

relationship_corr.round(3)

Rating Count       0.964
Rating in Stars    0.095
Discount %         0.028
Current Price     -0.001
Name: Sold Count, dtype: float64

In [4]:
# NumPy verification of the strongest relationship: Rating Count vs Sold Count
rating_sales = df[["Rating Count", "Sold Count"]].dropna()
rating_count_array = rating_sales["Rating Count"].to_numpy(dtype=float)
sold_count_array = rating_sales["Sold Count"].to_numpy(dtype=float)

numpy_rating_sales_corr = np.corrcoef(rating_count_array, sold_count_array)[0, 1]
print("NumPy correlation (Rating Count vs Sold Count):", round(float(numpy_rating_sales_corr), 3))

NumPy correlation (Rating Count vs Sold Count): 0.964


**Cross-check.** NumPy returns the same correlation of approximately **0.964** for `Rating Count` versus `Sold Count`. This confirms that the strong relationship is not the result of one particular pandas command.

**Interpretation.** `Rating Count` has by far the strongest relationship with sold count. This makes practical sense because popular products tend to accumulate both purchases and ratings. By contrast, current price, star rating itself and discount percentage have weak relationships with sold count in this snapshot.

The important distinction is between **rating score** and **rating volume**: a product having 4.6 instead of 4.4 stars does not separate sales very much, but a product having hundreds or thousands of ratings is strongly associated with higher reported sales.

## Module 4 output
NumPy confirms the main distribution statistics and independently verifies the strongest correlation. These results are interpreted as **associations**, not causal effects.